# Validation Test StartUp (Omega study) - Capillary Rise (benchmark test case within SFB 1194)

This notebook and the corresponding runsimulation and evaluation notebooks (CapillaryRise_SFB1194_Run.ipynb, CapillaryRise_SFB1194_Evaluation.ipynb) are part of published results (cf. 4.6) found in *A comparative study of transient capillary rise using direct numerical simulations* (https://doi.org/10.1016/j.apm.2020.04.020). 

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [3]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [4]:
//var myBatch = ExecutionQueues[1];
var myBatch = (userName.Equals(@"FDY\jenkinsci")) ? GetDefaultQueue() : ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [5]:
wmg.Init("CapillaryRisePaper", myBatch);

In [6]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [7]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [8]:
static var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: CapillaryRisePaper	CapillaryRise_Omega1_OmegaStudy_mesh8_StartUp	8/28/2018 5:24:26 PM	33e5c091...
#1: CapillaryRisePaper	CapillaryRise_Omega100_OmegaStudy_mesh8_StartUp	8/28/2018 5:25:49 PM	ccd0a03e...
#2: CapillaryRisePaper	CapillaryRise_Omega10_OmegaStudy_mesh8_StartUp	8/28/2018 5:25:03 PM	387c680e...
#3: CapillaryRisePaper	CapillaryRise_Omega05_OmegaStudy_mesh8_AMR1_StartUp	8/28/2018 5:23:25 PM	9dbc569e...
#4: CapillaryRisePaper	CapillaryRise_Omega01_OmegaStudy_mesh8_AMR1_StartUp*	8/30/2018 10:36:28 AM	32d62e31...
#5: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh2_StartUp	11/13/2018 5:27:47 PM	c572378f...
#6: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh1_StartUp	11/13/2018 5:30:03 PM	fda344a0...
#7: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh4_StartUp	11/13/2018 5:25:15 PM	1a788081...
#8: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh8_StartUp	11/13/2018 5:23:56 PM	d169b7fc...
#9: CapillaryRisePaper	CapillaryRise_Omega1_meshStudy_mesh16_Star

In [ ]:
// foreach (var s in sessions) {
//     if(s.Name.Contains("CapillaryRise_Omega01_OmegaStudy_mesh8_AMR1_startUp") && s.CreationTime.Date == new DateTime(2025, 11, 24)) {
//         s.Delete(true);
//         //Console.WriteLine($"Deleted session from {s.CreationTime.Date}");
//     }
// }

In [ ]:
// static var sessions = BoSSSshell.WorkflowMgm.Sessions;
// sessions

## Grid creation

In [9]:
static int[] Resolutions = new int[] { 1, 2, 4, 8, 16 };    
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

static double R = 5.0e-3;      // in cm
static double H = 4.0e-2;

bool useSymmetry = true;

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];

    string GridName;
    if(useSymmetry)
        GridName = $"CapillaryRise_{Res}x{8*Res}_halfTube";
    else
        GridName = $"CapillaryRise_{Res}x{8*Res}_fullTube";

    IGridInfo cachedGrid = BoSSSshell.WorkflowMgm.Grids.FirstOrDefault(grid => grid.Name == GridName);
    // cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] Xnodes;
        if(useSymmetry)
            Xnodes = GenericBlas.Linspace(0, R, Res + 1);
        else
            Xnodes = GenericBlas.Linspace(-R, R, (2 * Res) + 1);
        double[] Ynodes = GenericBlas.Linspace(0, H, 8 * Res + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);
    
        grd.EdgeTagNames.Add(1, "wall_lower");
        grd.EdgeTagNames.Add(2, "pressure_outlet_upper");

        if(useSymmetry)
            grd.EdgeTagNames.Add(3, "slipsymmetry_left");
        else
            grd.EdgeTagNames.Add(3, "navierslip_linear_left");

        grd.EdgeTagNames.Add(4, "navierslip_linear_right");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;
            if(Math.Abs(X[1]) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - H) <= 1.0e-8)
                et = 2;
            if(useSymmetry) {
                if(Math.Abs(X[0]) <= 1.0e-8)
                    et = 3;
            } else {
                if(Math.Abs(X[0] + R) <= 1.0e-8)
                    et = 3;
            }
            if(Math.Abs(X[0] - R) <= 1.0e-8)
                et = 4;

            return et;
        });     
        
        grd.Name = GridName;
        
        Grids[i] = BoSSSshell.WorkflowMgm.SaveGrid(grd);
        //Grids[i] = grd;
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Error: System.IO.FileNotFoundException: Could not find file '\\dc3\userspace\smuda\hpccluster\CapillaryRisePaper\timesteps\618e0ca4-ca53-4db5-8d4b-f9aba47c648b.ts'.
File name: '\\dc3\userspace\smuda\hpccluster\CapillaryRisePaper\timesteps\618e0ca4-ca53-4db5-8d4b-f9aba47c648b.ts'
   at BoSSS.Foundation.IO.StandardFsDriver.OpenFileExclusiveBlocking(Boolean create, String RelPath, Boolean ForceOverride) in C:\BoSSS-experimental\public\src\L2-foundation\BoSSS.Foundation\StandardFsDriver.cs:line 327
   at BoSSS.Foundation.IO.StandardFsDriver.GetTimestepStream(Boolean create, Guid sessionGuid) in C:\BoSSS-experimental\public\src\L2-foundation\BoSSS.Foundation\StandardFsDriver.cs:line 371
   at BoSSS.Foundation.IO.TimeStepDatabaseDriver.LoadTimestepInfo[T](Guid timestepGuid, ISessionInfo session, IDatabaseInfo database) in C:\BoSSS-experimental\public\src\L2-foundation\BoSSS.Foundation\DatabaseDriver_TimeStep.cs:line 40
   at BoSSS.Foundation.IO.DatabaseDriver.LoadTimestepInfo(Guid times

## Initial Values

In [10]:
Formula PhiFunc = new Formula(
    "Phi",
    false,
    "double Phi(double[] X) { return X[1] - 1e-2; } ");

## Physical settings

In [11]:
string[] setupS = new string[] {
    "Omega0.1", "Omega0.5", "Omega1", "Omega10", "Omega100"
};

In [12]:
public class PhysicalSettings {

    public double rhoA;
    public double muA;
    public double rhoB;
    public double muB;
    public double sigma;

    public double Lslip;
    public double betaS_A;
    public double betaS_B;

    public double betaL;
    public double theta_e;

    public Formula GravityValue;


    public PhysicalSettings(string setup) {

        muA = 0.01; 
        muB = 0.01 / 1000;

        Lslip = R / 5.0;
        betaS_A = (muA / Lslip);
        betaS_B = (muB / Lslip);

        betaL = 0;
        theta_e = 3.0 * Math.PI / 18.0;

        switch(setup) {
            case "Omega01": {

                rhoA = 1663.8;
                rhoB = 1663.8 / 1000;      
                sigma = 0.2;       // kg / s^2

                GravityValue = new Formula(
                    "GravY",
                    false,
                    "double GravY(double[] X) { return -1.04; } "
                );
                
                break;
            }
            case "Omega05": {

                rhoA = 133.0;
                rhoB = 133.0 / 1000;      
                sigma = 0.1;       // kg / s^2

                GravityValue = new Formula(
                    "GravY",
                    false,
                    "double GravY(double[] X) { return -6.51; } "
                );

                break;
            }
            case "Omega1": {

                rhoA = 83.1;
                rhoB = 83.1 / 1000;      
                sigma = 0.04;       // kg / s^2

                GravityValue = new Formula(
                    "GravY",
                    false,
                    "double GravY(double[] X) { return -4.17; } "
                );

                break;
            }
            case "Omega10": {

                rhoA = 3.3255;
                rhoB = 3.3255 / 1000;      
                sigma = 0.01;       // kg / s^2

                GravityValue = new Formula(
                    "GravY",
                    false,
                    "double GravY(double[] X) { return -26.042; } "
                );

                break;
            }
            case "Omega100": {

                rhoA = 0.33255;
                rhoB = 0.33255 / 1000;      
                sigma = 0.001;       // kg / s^2

                GravityValue = new Formula(
                    "GravY",
                    false,
                    "double GravY(double[] X) { return -26.042; } "
                );

                break;
            }
        }
    }
}

## Control settings

In [13]:
public class StudySettings {

    public string studyName;

    public string OmegaCase;
    public PhysicalSettings PhysicalParams;

    public int grdIdx;
    public int AMRlvl;

    public double t_end;
    public double dt;
    public int savePeriod;
    public int logPeriod;

    public StudySettings(string name, string testcase, int gridIndex, int AMRlevel, double tend, double timeStep, int savePer = 100, int logPer = 100) {
        studyName = name;
        OmegaCase = testcase;
        PhysicalParams = new PhysicalSettings(testcase);

        grdIdx = gridIndex;  
        AMRlvl = AMRlevel;   

        t_end = tend;
        dt = timeStep;
        double hmin = R / (Resolutions[gridIndex]) / (AMRlevel + 1);
        double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(PhysicalParams.rhoA, PhysicalParams.rhoB, PhysicalParams.sigma, hmin, 2+1);
        Console.WriteLine($"  dt (user) = {timeStep}, dt (capillary) = {dt_sigma}");

        savePeriod = savePer;   
        logPeriod = logPer;
    }

}

In [14]:
Resolutions

[ 1, 2, 4, 8, 16 ]

In [ ]:
List<StudySettings> allStudies = new List<StudySettings>();
// omega study  
allStudies.Add(new StudySettings("OmegaStudy", "Omega01", 3, 1, 0.4, 2e-5));    //omega0.1  AMR 1
allStudies.Add(new StudySettings("OmegaStudy", "Omega05", 3, 1, 0.2, 1e-5));    //omega0.1  AMR 1
allStudies.Add(new StudySettings("OmegaStudy", "Omega1", 3, 0, 0.2, 2.5e-5));    //omega0.1
allStudies.Add(new StudySettings("OmegaStudy", "Omega10", 3, 0, 0.08, 1e-5));    //omega0.1
allStudies.Add(new StudySettings("OmegaStudy", "Omega100", 3, 0, 0.4,  5e-5));    //omega0.1

In [16]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
foreach(var study in allStudies) {

var C = new XNSE_Control();

int k = 2;
C.SetDGdegree(k);

C.FieldOptions.Add(VariableNames.GravityY + "#A", new FieldOpts() {
    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
});
C.FieldOptions.Add(VariableNames.GravityY + "#B", new FieldOpts() {
    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
});

// physical parameters
// ===================
var pParams = study.PhysicalParams;

C.PhysicalParameters.rho_A = pParams.rhoA;
C.PhysicalParameters.mu_A = pParams.muA;
C.PhysicalParameters.rho_B = pParams.rhoB;
C.PhysicalParameters.mu_B = pParams.muB;

C.PhysicalParameters.Sigma = pParams.sigma;

C.PhysicalParameters.betaS_A = pParams.betaS_A;
C.PhysicalParameters.betaS_B = pParams.betaS_B;

C.PhysicalParameters.betaL = pParams.betaL;
C.PhysicalParameters.theta_e = pParams.theta_e;

C.PhysicalParameters.IncludeConvection = false;
C.PhysicalParameters.Material = true;

    
// set grid
// ========
C.SetGrid(Grids[study.grdIdx]);
    
// initial conditions
// ===================
C.AddInitialValue("Phi", PhiFunc);

C.AddInitialValue("GravityY#A", pParams.GravityValue);      
C.AddInitialValue("GravityY#B", pParams.GravityValue);

// boundary conditions
// ===================
C.AddBoundaryValue("wall_lower");
C.AddBoundaryValue("pressure_outlet_upper");

if(useSymmetry)
    C.AddBoundaryValue("slipsymmetry_left");
else
    C.AddBoundaryValue("navierslip_linear_left");

C.AddBoundaryValue("navierslip_linear_right");

C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_SlipLength;
C.PhysicalParameters.sliplength = pParams.Lslip;


// misc. solver options
// ====================
C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

C.CutCellQuadratureType = CutCellQuadratureMethod.Saye;
C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

C.LinearSolver = LinearSolverCode.direct_mumps.GetConfig(); 
C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 50;


// level-set
// =========
C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;

C.AdaptiveMeshRefinement = false;

if (study.AMRlvl > 0) {
    C.AdaptiveMeshRefinement = true;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowbandAtBoundary(new byte[] { 4 }) { useUnion = true, maxRefinementLevel = study.AMRlvl });
    C.AMR_startUpSweeps = study.AMRlvl;
}

// Timestepping
// ============
C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;

C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C.dtMin = study.dt;
C.dtMax = study.dt;
C.Endtime = study.t_end;
C.NoOfTimesteps = (int)(C.Endtime / C.dtMin);
C.saveperiod = study.savePeriod;
     
    
C.PostprocessingModules.Add(new BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.MovingContactLineLogging() { SolverStage = 2, LogPeriod = study.logPeriod });
    
if (study.AMRlvl > 0) {
    C.SessionName = $"CapillaryRise_{study.OmegaCase}_{study.studyName}_mesh{Resolutions[study.grdIdx]}_AMR{study.AMRlvl}_startUp";    
} else {
    C.SessionName = $"CapillaryRise_{study.OmegaCase}_{study.studyName}_mesh{Resolutions[study.grdIdx]}_startUp";  
}
    
Controls.Add(C);

}

In [21]:
foreach (var ctrl in Controls) {
    Console.WriteLine($"{ctrl.SessionName}");
    // Console.WriteLine($"{ctrl.SessionName}: dt = {ctrl.dtMin}; saveperiod = {ctrl.saveperiod}; logperiod = {ctrl.PostprocessingModules.Pick(0).LogPeriod}");
}

In [22]:
int NoCtrls = Controls.Count();
NoCtrls

5

## Launch job

In [ ]:
List<Job> jobs = new List<Job>();

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);

    jobs.Add(oneJob);
}

Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh8	3/23/2020 9:35:37 PM	bdd58f86..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh16	3/24/2020 2:29:40 PM	eed03bcf..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-W

## Wait for Completion and Check Job Status

In [ ]:
int NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress 
                                || job.Status == JobStatus.PendingInExecutionQueue
                                || job.Status == JobStatus.PreActivation).Count();
NoInProgress

0

In [ ]:
int maxDays = 1;
int waitingDays = 0;
while (NoInProgress > 0 && waitingDays < maxDays) {
    wmg.BlockUntilAllJobsTerminate(3600*24*1); // wait one day for the jobs to finish
    NoInProgress = jobs.Where(job => job.Status == JobStatus.InProgress).Count();
    waitingDays++;
}

In [ ]:
var FailedSess = jobs.Where(job => job.Status == JobStatus.FailedOrCanceled);
FailedSess

0

In [ ]:
int NoFailed = FailedSess.Count();
NoFailed

In [ ]:
var SuccessSess = jobs.Where(job => job.Status == JobStatus.FinishedSuccessful);
SuccessSess

In [ ]:
int NoSuccess = SuccessSess.Count();
NoSuccess

17

In [ ]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var job in jobs) {
        //var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(job.Name));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {job.Name}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {job.Name}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine("No restart session found. {job.Name} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {job.Name}!"); // should not happen
            } 
        }
    }

}

In [ ]:
//NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

In [ ]:
//NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");